# ChromSeek Loop Prediction Workflow

This notebook runs the pretrained ChromSeek loop prediction model on a 2.24 Mb DNA/Hi-C window. It follows the same interactive workflow as the Hi-C enhancement app:

1. Configure the sample and checkpoint.
2. Load and preprocess DNA, Hi-C, and loop labels.
3. Run loop probability inference.
4. Report AUROC/AUPRC and visualize annotated and predicted loops.

The bundled `sample_data.pt` is used by default, so no genome cache or mcool file is required.

In [ ]:
# ==========================================
# 1. Configuration & Setup
# ==========================================
from pathlib import Path

# Resolve the repository root whether opened from the root or a task directory.
for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (PROJECT_ROOT / 'hic_enhancement').is_dir() and (PROJECT_ROOT / 'downstreams').is_dir():
        break
else:
    raise FileNotFoundError('Could not locate the ChromSeek project root.')
TASK_DIR = PROJECT_ROOT / 'downstreams' / 'loop_prediction'

DATA_PATH = TASK_DIR / 'sample_data.pt'
CKPT_PATH = PROJECT_ROOT / 'checkpoints' / 'chromSeek_loop_prediction.pth'
PRED_THRESHOLD = 0.50
MIN_BIN_SEPARATION = 2  # Ignore the diagonal and immediate neighbours when scoring/plotting.

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Sample data not found: {DATA_PATH}')
if not CKPT_PATH.exists():
    raise FileNotFoundError(f'Loop checkpoint not found: {CKPT_PATH}')
print(f'Configured data: {DATA_PATH}')
print(f'Configured checkpoint: {CKPT_PATH}')
print(f'Prediction threshold: {PRED_THRESHOLD}')

In [ ]:
# ==========================================
# 2. Load Dependencies & Imports
# ==========================================
import os
import sys
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy import ndimage
from sklearn.metrics import roc_auc_score, average_precision_score

sys.path.insert(0, str(PROJECT_ROOT))
from downstreams.loop_prediction.model import LoopPredictionModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ==========================================
# 3. Load Sample DNA, Hi-C & Loop Labels
# ==========================================
data_list = torch.load(DATA_PATH, map_location='cpu', weights_only=False)
if not data_list:
    raise ValueError('The sample data file contains no examples.')
sample = data_list[0]

seq = sample['inputs']['seq']
hic = sample['inputs']['hic']
gt_loop = sample['targets']['loop'].squeeze().cpu().numpy().astype(np.int64)
raw_hic = np.asarray(sample['info'].get('raw_hic', hic.squeeze().cpu().numpy()))

print(f'DNA tensor: {tuple(seq.shape)}')
print(f'Hi-C tensor: {tuple(hic.shape)}')
print(f'Ground-truth loop mask: {gt_loop.shape}; positive bins: {int(gt_loop.sum())}')
print(f'Raw Hi-C matrix: {raw_hic.shape}')

In [ ]:
# ==========================================
# 4. Prepare Model Inputs
# ==========================================
t_seq = seq.float()
if t_seq.ndim == 2:
    t_seq = t_seq.unsqueeze(0)
t_hic = hic.float()
if t_hic.ndim == 3:
    t_hic = t_hic.unsqueeze(0)

# The bundled sample already contains the training-time log1p/max-normalized Hi-C tensor.
# Only sanitize it here; applying log1p a second time would change the input distribution.
t_hic = torch.nan_to_num(t_hic, nan=0.0, posinf=0.0, neginf=0.0).clamp_min(0)
t_seq = t_seq.to(device)
t_hic = t_hic.to(device)
print(f'Model DNA input: {tuple(t_seq.shape)}')
print(f'Model Hi-C input: {tuple(t_hic.shape)}')

In [ ]:
# ==========================================
# 5. Load Pretrained Loop Model
# ==========================================
model = LoopPredictionModel(pretrained_path=None).to(device)
state_dict = torch.load(CKPT_PATH, map_location=device, weights_only=True)
# Checkpoints trained with DataParallel may contain a 'module.' prefix.
if state_dict and next(iter(state_dict)).startswith('module.'):
    state_dict = {key.removeprefix('module.'): value for key, value in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()
print('Loop prediction model is ready.')

In [ ]:
# ==========================================
# 6. Run Loop Probability Inference
# ==========================================
amp_enabled = device.type == 'cuda'
with torch.no_grad():
    with torch.autocast(device_type=device.type, enabled=amp_enabled):
        logits = model(t_seq, t_hic)
    pred_prob = F.softmax(logits, dim=1)[:, 1].squeeze(0).float().cpu().numpy()

print(f'Logits shape: {tuple(logits.shape)}')
print(f'Probability range: [{pred_prob.min():.4f}, {pred_prob.max():.4f}]')

In [ ]:
# ==========================================
# 7. Compute Loop Metrics
# ==========================================
if gt_loop.shape != pred_prob.shape:
    raise ValueError(f'Label/prediction shape mismatch: {gt_loop.shape} vs {pred_prob.shape}')
triu_indices = np.triu_indices_from(gt_loop, k=MIN_BIN_SEPARATION)
gt_flat = gt_loop[triu_indices].astype(int)
pred_flat = pred_prob[triu_indices]
if np.unique(gt_flat).size > 1:
    loop_auroc = roc_auc_score(gt_flat, pred_flat)
    loop_auprc = average_precision_score(gt_flat, pred_flat)
else:
    loop_auroc = loop_auprc = 0.0
print(f'Loop AUROC: {loop_auroc:.4f}')
print(f'Loop AUPRC: {loop_auprc:.4f}')

In [ ]:
# ==========================================
# 8. Extract Loop Peaks
# ==========================================
def extract_loop_peaks(prob_map, threshold=0.5, min_separation=2):
    mask = (prob_map >= threshold).astype(np.uint8)
    labels, count = ndimage.label(mask)
    peaks = []
    for region_id in range(1, count + 1):
        coords = np.argwhere(labels == region_id)
        coords = [tuple(c) for c in coords if c[1] - c[0] >= min_separation]
        if coords:
            peaks.append(max(coords, key=lambda c: prob_map[c]))
    return peaks

def extract_ground_truth_peaks(binary_map, min_separation=2):
    labels, count = ndimage.label(binary_map.astype(np.uint8))
    peaks = []
    for region_id in range(1, count + 1):
        coords = np.argwhere(labels == region_id)
        coords = [tuple(c) for c in coords if c[1] - c[0] >= min_separation]
        if coords:
            peaks.append(coords[len(coords) // 2])
    return peaks

pred_peaks = extract_loop_peaks(pred_prob, PRED_THRESHOLD, MIN_BIN_SEPARATION)
true_peaks = extract_ground_truth_peaks(gt_loop, MIN_BIN_SEPARATION)
print(f'Predicted loop peaks (threshold={PRED_THRESHOLD:.2f}): {len(pred_peaks)}')
print(f'Ground-truth loop peaks: {len(true_peaks)}')

In [ ]:
# ==========================================
# 9. Visualize Loop Predictions
# ==========================================
def plot_hic_with_peaks(ax, matrix, peaks, title, color):
    display = np.log1p(np.nan_to_num(matrix, nan=0.0).clip(min=0))
    positive = display[display > 0]
    vmax = np.percentile(positive, 80) if positive.size else 1.0
    ax.imshow(display, cmap='Reds', vmin=0, vmax=max(vmax, 1e-6), origin='upper')
    if peaks:
        rows, cols = zip(*peaks)
        ax.scatter(cols, rows, facecolors='none', edgecolors=color, s=90, linewidths=2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Hi-C bins (10 kb)')
    ax.set_ylabel('Hi-C bins (10 kb)')

fig, axes = plt.subplots(1, 3, figsize=(19, 6))
plot_hic_with_peaks(axes[0], raw_hic, true_peaks, 'Ground-truth loops', 'deepskyblue')
plot_hic_with_peaks(axes[1], raw_hic, pred_peaks, 'Predicted loops', 'lime')
im = axes[2].imshow(pred_prob, cmap='viridis', vmin=0, vmax=1, origin='upper')
axes[2].set_title('Predicted loop probability', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Hi-C bins (10 kb)')
axes[2].set_ylabel('Hi-C bins (10 kb)')
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 10. Explore Threshold Sensitivity (Optional)
# ==========================================
for threshold in (0.30, 0.50, 0.70, 0.90):
    count = len(extract_loop_peaks(pred_prob, threshold, MIN_BIN_SEPARATION))
    print(f'threshold={threshold:.2f}: {count} predicted peaks')

print('Workflow complete. Change PRED_THRESHOLD and rerun the peak/visualization cells to inspect another operating point.')